In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

DATA_PATH = Path("../data/raw/WineQT.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality,Id
0,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,0
1,7.8,0.88,0.00,2.6,0.098,25.0,67.0,0.9968,3.20,0.68,9.8,5,1
2,7.8,0.76,0.04,2.3,0.092,15.0,54.0,0.9970,3.26,0.65,9.8,5,2
3,11.2,0.28,0.56,1.9,0.075,17.0,60.0,0.9980,3.16,0.58,9.8,6,3
4,7.4,0.70,0.00,1.9,0.076,11.0,34.0,0.9978,3.51,0.56,9.4,5,4


In [2]:
df.shape

(1143, 13)

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1143 entries, 0 to 1142
Data columns (total 13 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   fixed acidity         1143 non-null   float64
 1   volatile acidity      1143 non-null   float64
 2   citric acid           1143 non-null   float64
 3   residual sugar        1143 non-null   float64
 4   chlorides             1143 non-null   float64
 5   free sulfur dioxide   1143 non-null   float64
 6   total sulfur dioxide  1143 non-null   float64
 7   density               1143 non-null   float64
 8   pH                    1143 non-null   float64
 9   sulphates             1143 non-null   float64
 10  alcohol               1143 non-null   float64
 11  quality               1143 non-null   int64  
 12  Id                    1143 non-null   int64  
dtypes: float64(11), int64(2)
memory usage: 116.2 KB


In [4]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
fixed acidity,1143.0,8.311111,1.747595,4.60000,7.10000,7.90000,9.100000,15.90000
volatile acidity,1143.0,0.531339,0.179633,0.12000,0.39250,0.52000,0.640000,1.58000
citric acid,1143.0,0.268364,0.196686,0.00000,0.09000,0.25000,0.420000,1.00000
residual sugar,1143.0,2.532152,1.355917,0.90000,1.90000,2.20000,2.600000,15.50000
chlorides,1143.0,0.086933,0.047267,0.01200,0.07000,0.07900,0.090000,0.61100
free sulfur dioxide,1143.0,15.615486,10.250486,1.00000,7.00000,13.00000,21.000000,68.00000
total sulfur dioxide,1143.0,45.914698,32.782130,6.00000,21.00000,37.00000,61.000000,289.00000
density,1143.0,0.996730,0.001925,0.99007,0.99557,0.99668,0.997845,1.00369
pH,1143.0,3.311015,0.156664,2.74000,3.20500,3.31000,3.400000,4.01000
sulphates,1143.0,0.657708,0.170399,0.33000,0.55000,0.62000,0.730000,2.00000


In [5]:
df.isna().sum()

fixed acidity           0
volatile acidity        0
citric acid             0
residual sugar          0
chlorides               0
free sulfur dioxide     0
total sulfur dioxide    0
density                 0
pH                      0
sulphates               0
alcohol                 0
quality                 0
Id                      0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
df.columns

Index(['fixed acidity', 'volatile acidity', 'citric acid', 'residual sugar',
       'chlorides', 'free sulfur dioxide', 'total sulfur dioxide', 'density',
       'pH', 'sulphates', 'alcohol', 'quality', 'Id'],
      dtype='str')

In [8]:
df["quality"].value_counts().sort_index()

quality
3      6
4     33
5    483
6    462
7    143
8     16
Name: count, dtype: int64

In [9]:
df["quality"].value_counts(normalize=True).sort_index()

quality
3    0.005249
4    0.028871
5    0.422572
6    0.404199
7    0.125109
8    0.013998
Name: proportion, dtype: float64

In [10]:
df["quality"].describe()

count    1143.000000
mean        5.657043
std         0.805824
min         3.000000
25%         5.000000
50%         6.000000
75%         6.000000
max         8.000000
Name: quality, dtype: float64

In [11]:
df["quality"].nunique()

6

In [12]:
df["quality"].unique()

array([5, 6, 7, 4, 8, 3])

In [15]:
binary_target = (df["quality"] >= 7).astype(int)
binary_target.value_counts().sort_index()

quality
0    984
1    159
Name: count, dtype: int64

In [16]:
binary_target.value_counts(normalize=True).sort_index()

quality
0    0.860892
1    0.139108
Name: proportion, dtype: float64

## Initial task framing

The target `quality` is an ordinal target with 6 observed values: 3, 4, 5, 6, 7, 8.

Although it is stored as an integer, it should not be treated as a purely continuous variable without caution, because the values represent ordered quality scores rather than physical measurements.

Observed target distribution is imbalanced:
- Most wines have quality 5 or 6.
- Very low and very high quality classes are rare.
- Classes 3, 4, and 8 have very few observations.

Possible task formulations:

1. Regression:
   - Predict the numeric quality score.
   - Possible metrics: MAE, RMSE.
   - Pros: respects ordering partly and avoids rare-class classification issues.
   - Cons: assumes numeric distances between scores are meaningful.

2. Multiclass classification:
   - Predict exact quality class.
   - Possible metrics: macro F1, weighted F1, balanced accuracy.
   - Pros: treats quality as discrete labels.
   - Cons: severe class imbalance; rare classes may be poorly predicted.

3. Binary classification:
   - Example: good wine = quality >= 7.
   - Possible metrics: F1, ROC-AUC, PR-AUC, balanced accuracy.
   - Pros: simpler first classification framing.
   - Cons: loses ordinal information and creates an imbalanced binary target.

In [17]:
quality_distribution = (
    df["quality"]
    .value_counts()
    .sort_index()
    .rename_axis("quality")
    .reset_index(name="count")
)

quality_distribution["proportion"] = (
    quality_distribution["count"] / quality_distribution["count"].sum()
)

quality_distribution

,quality,count,proportion
0,3,6,0.005249
1,4,33,0.028871
2,5,483,0.422572
3,6,462,0.404199
4,7,143,0.125109
5,8,16,0.013998


In [18]:
binary_distribution = (
    binary_target
    .value_counts()
    .sort_index()
    .rename_axis("is_good_quality")
    .reset_index(name="count")
)

binary_distribution["proportion"] = (
    binary_distribution["count"] / binary_distribution["count"].sum()
)

binary_distribution

,is_good_quality,count,proportion
0,0,984,0.860892
1,1,159,0.139108
